## Demo: Multifidelity UQ for OPF

Demo for running ACV-MRP on the OPF multifidelity model pair.

* HF model: ACOPF with intertemporal generator coupling.

* LF model: DCOPF (default) or copperplate dispatch with the same scenario batches.

The two models share:
- finite scenario population,
- one scenario sampler,
- one first-stage decision space.

The output is a point estimate and confidence interval for an upper bound on the optimality gap, using a low-fidelity control variate.

In [1]:
from sparow.conf_intervals.options import UQOptions
from sparow.conf_intervals.acv_mrp import ACVMRP
from sparow.conf_intervals.evaluate_true_optimality_gap import TrueOptimalityGapEvaluator
from sparow.conf_intervals.experiment_helpers import run_acvmrp

# This is the function you have to define for your problem instance
from uq_opf import get_model_ensemble_for_uq

The code below outlines the step-by-step workflow.

In [2]:
# ------------------------------------------------------------------
# Step 1: Build the shared HF/LF ensemble
# ------------------------------------------------------------------
ensemble = get_model_ensemble_for_uq(
    model_name="HF",              # ignored; kept for interface consistency
    seed=12345,
    with_replacement=True,
    lf_model_type="dcopf",        # alternatives: "copperplate"
)

hf_model = ensemble.high_fidelity_model()
lf_model = ensemble.low_fidelity_model()

In [3]:
# ------------------------------------------------------------------
# Step 2: Generate one candidate first-stage solution xhat
# ------------------------------------------------------------------
# We first solve one HF SAA on a random subset of the finite scenario population and
# then extract the resulting first-stage vector as a candidate to evaluate.

n_xhat = 1 # choose a small batch size for candidate generation
xhat_replication_id = 999  # fixed id so the sampled batch is reproducible

xhat_scenarios = hf_model.draw_batch_of_scenarios(n=n_xhat, replication_id=xhat_replication_id,)

solved_hf = hf_model.solve_saa(
    sampled_scenarios=xhat_scenarios,
    solver_name="ipopt", # use nonlinear solver because ACOPF contains nonlinear, nonconvex expressions
    solver_options=None,
)

xhat = hf_model.get_first_stage_solution(solved_hf)

print("\nCandidate first-stage solution xhat extracted from one HF SAA on a random subset:")
print(f"Number of scenarios used to generate xhat: {n_xhat}")
for k, v in xhat.items():
    print(f"  {k}: {v}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP



Candidate first-stage solution xhat extracted from one HF SAA on a random subset:
Number of scenarios used to generate xhat: 1
  time_periods[1].m.pg['1']: 2.1820366644023546
  time_periods[1].m.pg['2']: 0.7167423214444644
  time_periods[1].m.pg['3']: 0.0
  time_periods[1].m.pg['4']: 0.0
  time_periods[1].m.pg['5']: 0.0
  time_periods[1].m.pg['6']: 0.0


In [4]:
# ------------------------------------------------------------------
# Step 3: Configure ACV-MRP
# ------------------------------------------------------------------
options = UQOptions(
    n=5,                 # batch size per replication
    m=10,                # paired HF/LF replications
    M=10,                # additional LF-only replications
    alpha=0.05,          # one-sided confidence level
    seed=12345,
    with_replacement=True,
    solver_name="ipopt",
    verbose=True,
)

In [5]:
# ------------------------------------------------------------------
# Step 4: Run ACV-MRP
# ------------------------------------------------------------------
acv_algorithm = ACVMRP(
    hf_model=hf_model,
    lf_model=lf_model,
    options=options,
)

results = acv_algorithm.run(xhat=xhat)

print("\nACV-MRP results:")
print(f"ACV-MRP Point Estimate: {results['point_estimate']}")
print(f"ACV-MRP Confidence Interval: [{results['ci_lower']}, {results['ci_upper']}]")
print(f"Estimated control variate coefficient: {results['control_variate_coefficient']}")
print(f"Estimated sample correlation: {results['sample_correlation']}")
print("\n")
print(f"Point estimate using only high-fidelity model (no paired or extra low-fidelity evals): {results['point_estimate_hf_only']}")
print(f"Variance reduction factor from spending additional computation on low-fidelity evals: {results['variance_reduction_factor']}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Running ACV-MRP with m=10, M=10, n=5
Using precomputed superset of scenarios for nested sampling scheme: False
Running paired ACV-MRP replication 1/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for high-fidelity paired replication 1: F_nk = 4871.6797386239705
Gap estimate for low-fidelity paired replication 1 : G_nk = 4071.0126340207353
Running paired ACV-MRP replication 2/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


...

Gap estimate for low-fidelity additional replication 18 : G_nk = 4270.126673158375
Running LF-only replication 9/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for low-fidelity additional replication 19 : G_nk = 3992.846995147738
Running LF-only replication 10/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP


Gap estimate for low-fidelity additional replication 20 : G_nk = 6420.479479431662

ACV-MRP results:
ACV-MRP Point Estimate: 4669.6113194302625
ACV-MRP Confidence Interval: [0.0, 5647.821005755383]
Estimated control variate coefficient: 1.1267621131823038
Estimated sample correlation: 0.6006263752960613


Point estimate using only high-fidelity model (no paired or extra low-fidelity evals): 4834.56522869018
Variance reduction factor from spending additional computation on low-fidelity evals: 12.200716743888837


Note that the helper function below, `run_acvmrp` can also be used to run the ACVMRP algorithm. It accepts the following as arguments:
- a given model ensemble (STEP 1), 
- a candidate first-stage solution, xhat (STEP 2), and 
- a set of options/ parameters (STEP 3). 

It then runs the algorithm (STEP 4) and outputs results. This function is primarily intended for ease of use in numerical experiments.

In [6]:
results_from_helper_function = run_acvmrp(
    ensemble=ensemble,
    xhat=xhat,
    n=5,                 # batch size per replication
    m=10,                # paired HF/LF replications
    M=10,                # additional LF-only replications
    alpha=0.05,          # one-sided confidence level
    seed=12345,
    with_replacement=True,
    solver_name="ipopt",
    solver_options=None,
    verbose=True,
)

print("\nACV-MRP results from helper function:")
print(f"ACV-MRP Point Estimate: {results_from_helper_function['point_estimate']}")
print(f"ACV-MRP Confidence Interval: [{results_from_helper_function['ci_lower']}, {results_from_helper_function['ci_upper']}]")
print(f"Estimated control variate coefficient: {results_from_helper_function['control_variate_coefficient']}")
print(f"Estimated sample correlation: {results_from_helper_function['sample_correlation']}")
print("\n")
print(f"Point estimate using only high-fidelity model (no paired or extra low-fidelity evals): {results_from_helper_function['point_estimate_hf_only']}")
print(f"Variance reduction factor from spending additional computation on low-fidelity evals: {results_from_helper_function['variance_reduction_factor']}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Running ACV-MRP with m=10, M=10, n=5
Using precomputed superset of scenarios for nested sampling scheme: False
Running paired ACV-MRP replication 1/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for high-fidelity paired replication 1: F_nk = 4871.6797386239705
Gap estimate for low-fidelity paired replication 1 : G_nk = 4071.0126340207353
Running paired ACV-MRP replication 2/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


...

Gap estimate for low-fidelity additional replication 18 : G_nk = 4270.126673158375
Running LF-only replication 9/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for low-fidelity additional replication 19 : G_nk = 3992.846995147738
Running LF-only replication 10/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP


Gap estimate for low-fidelity additional replication 20 : G_nk = 6420.479479431662

ACV-MRP results from helper function:
ACV-MRP Point Estimate: 4669.6113194302625
ACV-MRP Confidence Interval: [0.0, 5647.821005755383]
Estimated control variate coefficient: 1.1267621131823038
Estimated sample correlation: 0.6006263752960613


Point estimate using only high-fidelity model (no paired or extra low-fidelity evals): 4834.56522869018
Variance reduction factor from spending additional computation on low-fidelity evals: 12.200716743888837


**Optional**: If your problem instance is small enough to solve over all population scenarios within a reasonable amount of time, you can compute the true optimality gap for your candidate solution as a benchmark. 

In [7]:
# ------------------------------------------------------------------
# Step 5: Optional finite-population benchmark
# ------------------------------------------------------------------
# This computes the exact finite-population quantities over the stored
# scenario population for the HF model, which is useful for debugging
# and small-scale numerical validation.
true_gap_evaluator = TrueOptimalityGapEvaluator(
    model=hf_model,
    solver_name="ipopt",
    solver_options=None,
)

true_gap_results = true_gap_evaluator.compute_true_gap(xhat=xhat)

print("\nTrue finite-population HF quantities:")
print(f"True optimal value: {true_gap_results['true_optimal_value']}")
print(f"xhat true value: {true_gap_results['xhat_true_value']}")
print(f"True optimality gap: {true_gap_results['true_gap']}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP



True finite-population HF quantities:
True optimal value: 58593.92432766313
xhat true value: 63780.21665537743
True optimality gap: 5186.292327714298
